In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# Display settings
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

# Project paths
PROJECT_ROOT = Path("..").resolve()
PORTFOLIO_ROOT = PROJECT_ROOT.parents[1]

PROJECT_02 = PORTFOLIO_ROOT / "projects" / "02_microct_subvolume_3d_analysis"
PROJECT_03 = PORTFOLIO_ROOT / "projects" / "03_3d_microct_process_monitoring"

OUTPUT_DIR = PROJECT_ROOT / "data" / "dashboard_exports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project 05 root:", PROJECT_ROOT)
print("Portfolio root:", PORTFOLIO_ROOT)
print("Project 02 path exists:", PROJECT_02.exists())
print("Project 03 path exists:", PROJECT_03.exists())
print("Output directory:", OUTPUT_DIR)

Project 05 root: C:\Users\Peter\ai-imaging-portfolio\projects\05_microct_quality_monitoring_powerbi
Portfolio root: C:\Users\Peter\ai-imaging-portfolio
Project 02 path exists: True
Project 03 path exists: True
Output directory: C:\Users\Peter\ai-imaging-portfolio\projects\05_microct_quality_monitoring_powerbi\data\dashboard_exports


## 1. Inspect available outputs from previous projects

Before loading data, we inspect the available output files from Project 02 and Project 03.

This step helps verify which CSV, NPY and image files can be reused for the Power BI dashboard.

In [2]:
# Inspect relevant output folders from previous projects

candidate_folders = [
    PROJECT_02 / "data",
    PROJECT_02 / "outputs",
    PROJECT_02 / "results",
    PROJECT_02 / "figures",
    PROJECT_03 / "data",
    PROJECT_03 / "outputs",
    PROJECT_03 / "results",
    PROJECT_03 / "figures",
]

for folder in candidate_folders:
    print("\n" + "=" * 100)
    print(folder)
    print("Exists:", folder.exists())
    
    if folder.exists():
        files = sorted([p for p in folder.rglob("*") if p.is_file()])
        print(f"Number of files: {len(files)}")
        
        for file in files[:40]:
            print(" -", file.relative_to(folder))
        
        if len(files) > 40:
            print(f" ... and {len(files) - 40} more files")


C:\Users\Peter\ai-imaging-portfolio\projects\02_microct_subvolume_3d_analysis\data
Exists: True
Number of files: 440
 - processed\material_mask.npy
 - processed\phase_labels_3class.npy
 - processed\pores_mask.npy
 - processed\resin_mask.npy
 - processed\resin_mask_refined_ge50.npy
 - raw\DatosSubV\DatosSubV0000.tif
 - raw\DatosSubV\DatosSubV0001.tif
 - raw\DatosSubV\DatosSubV0002.tif
 - raw\DatosSubV\DatosSubV0003.tif
 - raw\DatosSubV\DatosSubV0004.tif
 - raw\DatosSubV\DatosSubV0005.tif
 - raw\DatosSubV\DatosSubV0006.tif
 - raw\DatosSubV\DatosSubV0007.tif
 - raw\DatosSubV\DatosSubV0008.tif
 - raw\DatosSubV\DatosSubV0009.tif
 - raw\DatosSubV\DatosSubV0010.tif
 - raw\DatosSubV\DatosSubV0011.tif
 - raw\DatosSubV\DatosSubV0012.tif
 - raw\DatosSubV\DatosSubV0013.tif
 - raw\DatosSubV\DatosSubV0014.tif
 - raw\DatosSubV\DatosSubV0015.tif
 - raw\DatosSubV\DatosSubV0016.tif
 - raw\DatosSubV\DatosSubV0017.tif
 - raw\DatosSubV\DatosSubV0018.tif
 - raw\DatosSubV\DatosSubV0019.tif
 - raw\DatosSubV\

## 2. Load main input tables

The dashboard will reuse two main sources:

- Project 03 outputs:
  - `features_by_slice.csv`
  - `anomaly_results.csv`

- Project 02 results:
  - `slice_by_slice_phase_fractions.csv`
  - `phase_fraction_descriptive_statistics.csv`
  - `comparison_object_counts_2d_vs_3d.csv`
  - `comparison_equivalent_diameter_2d_vs_3d.csv`
  - `components_3d_summary_by_phase.csv`
  - `final_project_summary.csv`

These tables provide the quantitative basis for the Power BI dashboard.

In [3]:
# Define input file paths

project03_outputs = PROJECT_03 / "outputs"
project02_results = PROJECT_02 / "results"

input_files = {
    "features_by_slice": project03_outputs / "features_by_slice.csv",
    "anomaly_results": project03_outputs / "anomaly_results.csv",
    "slice_phase_fractions": project02_results / "slice_by_slice_phase_fractions.csv",
    "phase_fraction_stats": project02_results / "phase_fraction_descriptive_statistics.csv",
    "object_counts_2d_vs_3d": project02_results / "comparison_object_counts_2d_vs_3d.csv",
    "diameter_2d_vs_3d": project02_results / "comparison_equivalent_diameter_2d_vs_3d.csv",
    "components_3d_summary": project02_results / "components_3d_summary_by_phase.csv",
    "final_project_summary": project02_results / "final_project_summary.csv",
}

# Check that all selected files exist
for name, path in input_files.items():
    print(f"{name:30s} | exists: {path.exists()} | {path}")

features_by_slice              | exists: True | C:\Users\Peter\ai-imaging-portfolio\projects\03_3d_microct_process_monitoring\outputs\features_by_slice.csv
anomaly_results                | exists: True | C:\Users\Peter\ai-imaging-portfolio\projects\03_3d_microct_process_monitoring\outputs\anomaly_results.csv
slice_phase_fractions          | exists: True | C:\Users\Peter\ai-imaging-portfolio\projects\02_microct_subvolume_3d_analysis\results\slice_by_slice_phase_fractions.csv
phase_fraction_stats           | exists: True | C:\Users\Peter\ai-imaging-portfolio\projects\02_microct_subvolume_3d_analysis\results\phase_fraction_descriptive_statistics.csv
object_counts_2d_vs_3d         | exists: True | C:\Users\Peter\ai-imaging-portfolio\projects\02_microct_subvolume_3d_analysis\results\comparison_object_counts_2d_vs_3d.csv
diameter_2d_vs_3d              | exists: True | C:\Users\Peter\ai-imaging-portfolio\projects\02_microct_subvolume_3d_analysis\results\comparison_equivalent_diameter_2d_vs_3d

In [4]:
# Load selected CSV files

tables = {}

for name, path in input_files.items():
    tables[name] = pd.read_csv(path)
    print("=" * 100)
    print(name)
    print("Shape:", tables[name].shape)
    print("Columns:")
    print(list(tables[name].columns))
    display(tables[name].head())

features_by_slice
Shape: (435, 22)
Columns:
['slice_id', 'z_um', 'pore_fraction', 'resin_fraction', 'matrix_fraction', 'assigned_fraction', 'mean_intensity', 'std_intensity', 'mean_intensity_pores', 'mean_intensity_resin', 'mean_intensity_matrix', 'pore_object_count', 'mean_pore_area_um2', 'median_pore_area_um2', 'mean_pore_equivalent_diameter_um', 'median_pore_equivalent_diameter_um', 'resin_object_count', 'mean_resin_area_um2', 'median_resin_area_um2', 'mean_resin_equivalent_diameter_um', 'median_resin_equivalent_diameter_um', 'heterogeneity_index']


,slice_id,z_um,pore_fraction,resin_fraction,matrix_fraction,assigned_fraction,mean_intensity,std_intensity,mean_intensity_pores,mean_intensity_resin,mean_intensity_matrix,pore_object_count,mean_pore_area_um2,median_pore_area_um2,mean_pore_equivalent_diameter_um,median_pore_equivalent_diameter_um,resin_object_count,mean_resin_area_um2,median_resin_area_um2,mean_resin_equivalent_diameter_um,median_resin_equivalent_diameter_um,heterogeneity_index
0,0,0.0,0.056875,0.303065,0.622066,0.982006,162.006452,34.223109,76.737897,136.006699,182.515447,250,52352.000000,19200.0,216.142166,156.352804,592,117805.405405,12800.0,233.533527,127.66153,0.231376
1,1,80.0,0.049366,0.302342,0.638364,0.990071,163.499583,33.344875,77.197183,136.707571,182.891997,251,45258.964143,12800.0,200.100077,127.661530,769,90473.862159,12800.0,216.507405,127.66153,0.241253
2,2,160.0,0.040800,0.294249,0.657276,0.992324,165.032206,32.662106,77.162236,136.070321,183.481699,261,35972.413793,12800.0,179.854408,127.661530,755,89684.768212,12800.0,221.468784,127.66153,0.252997
3,3,240.0,0.033513,0.284570,0.673963,0.992046,166.618951,32.181359,77.149378,135.372068,184.301490,249,30971.887550,12800.0,171.086291,127.661530,704,93018.181818,12800.0,225.478737,127.66153,0.263488
4,4,320.0,0.028952,0.265491,0.697630,0.992074,168.204917,32.126347,77.017291,133.324743,185.328058,260,25624.615385,12800.0,155.146942,127.661530,672,90914.285714,12800.0,234.745590,127.66153,0.276853


anomaly_results
Shape: (435, 17)
Columns:
['slice_id', 'z_um', 'pore_fraction', 'resin_fraction', 'matrix_fraction', 'mean_intensity', 'std_intensity', 'pore_object_count', 'mean_pore_equivalent_diameter_um', 'resin_object_count', 'mean_resin_equivalent_diameter_um', 'heterogeneity_index', 'PC1', 'PC2', 'is_anomaly', 'anomaly_score', 'anomaly_category']


,slice_id,z_um,pore_fraction,resin_fraction,matrix_fraction,mean_intensity,std_intensity,pore_object_count,mean_pore_equivalent_diameter_um,resin_object_count,mean_resin_equivalent_diameter_um,heterogeneity_index,PC1,PC2,is_anomaly,anomaly_score,anomaly_category
0,0,0.0,0.056875,0.303065,0.622066,162.006452,34.223109,250,216.142166,592,233.533527,0.231376,-16.003458,-2.397432,True,0.232357,high_porosity+resin_rich+resin_fragmented+larg...
1,1,80.0,0.049366,0.302342,0.638364,163.499583,33.344875,251,200.100077,769,216.507405,0.241253,-16.159629,-4.010913,True,0.224484,high_porosity+resin_rich+resin_fragmented+larg...
2,2,160.0,0.040800,0.294249,0.657276,165.032206,32.662106,261,179.854408,755,221.468784,0.252997,-14.789948,-4.687371,True,0.208778,high_porosity+resin_rich+resin_fragmented+larg...
3,3,240.0,0.033513,0.284570,0.673963,166.618951,32.181359,249,171.086291,704,225.478737,0.263488,-13.171530,-4.773668,True,0.180866,high_porosity+resin_rich+resin_fragmented
4,4,320.0,0.028952,0.265491,0.697630,168.204917,32.126347,260,155.146942,672,234.745590,0.276853,-11.460013,-4.886507,True,0.164346,high_porosity+resin_rich+resin_fragmented


slice_phase_fractions
Shape: (435, 6)
Columns:
['z_index', 'z_position_mm', 'pores_fraction', 'resin_fraction', 'material_fraction', 'total_fraction']


...

,phase,mean_equivalent_diameter_2d_um,median_equivalent_diameter_2d_um,std_equivalent_diameter_2d_um,mean_equivalent_diameter_3d_um,median_equivalent_diameter_3d_um,std_equivalent_diameter_3d_um
0,pores,362.344433,270.811000,331.069805,570.020004,496.280393,305.929826
1,resin,499.255427,469.058411,225.235480,686.088833,613.377414,389.979221


components_3d_summary
Shape: (2, 16)
Columns:
['phase', 'number_of_objects', 'mean_volume_voxels', 'median_volume_voxels', 'std_volume_voxels', 'mean_volume_mm3', 'median_volume_mm3', 'mean_equivalent_diameter_um', 'median_equivalent_diameter_um', 'std_equivalent_diameter_um', 'mean_elongation_bbox', 'median_elongation_bbox', 'mean_extent_3d', 'median_extent_3d', 'mean_intensity', 'median_intensity']


,phase,number_of_objects,mean_volume_voxels,median_volume_voxels,std_volume_voxels,mean_volume_mm3,median_volume_mm3,mean_equivalent_diameter_um,median_equivalent_diameter_um,std_equivalent_diameter_um,mean_elongation_bbox,median_elongation_bbox,mean_extent_3d,median_extent_3d,mean_intensity,median_intensity
0,pores,232,536.232759,125.0,3249.312729,0.274551,0.064000,570.020004,496.280393,305.929826,1.272622,1.166667,0.435776,0.492470,61.922860,58.269509
1,resin,1757,1810.328970,236.0,54766.772829,0.926888,0.120832,686.088833,613.377414,389.979221,1.499497,1.363636,0.324337,0.322751,132.133508,130.658038


final_project_summary
Shape: (13, 4)
Columns:
['category', 'metric', 'value', 'unit']


,category,metric,value,unit
0,phase_fraction,pores_fraction_global,0.012330,fraction
1,phase_fraction,resin_fraction_global,0.208809,fraction
2,phase_fraction,material_fraction_global,0.778862,fraction
3,2d_components,pores_number_of_2d_objects,5280.000000,objects
4,2d_components,pores_median_equivalent_diameter_2d,270.811000,um


## 3. Create slice dimension table

The `dim_slice` table provides one row per tomographic slice.

It will be used as the main dimension table in Power BI to filter all slice-level measurements.

Main fields:

- `slice_id`: slice identifier.
- `z_um`: slice position in micrometers.
- `z_mm`: slice position in millimeters.
- `relative_depth_percent`: normalized depth position from 0 to 100%.
- `slice_zone`: coarse location within the volume.

In [5]:
# Create dim_slice table from features_by_slice

features_by_slice = tables["features_by_slice"].copy()
anomaly_results = tables["anomaly_results"].copy()

dim_slice = features_by_slice[["slice_id", "z_um"]].copy()

# Add depth in millimeters
dim_slice["z_mm"] = dim_slice["z_um"] / 1000

# Add relative depth from 0 to 100%
z_min = dim_slice["z_um"].min()
z_max = dim_slice["z_um"].max()

dim_slice["relative_depth_percent"] = (
    (dim_slice["z_um"] - z_min) / (z_max - z_min) * 100
)

# Add coarse slice zones for dashboard grouping
dim_slice["slice_zone"] = pd.cut(
    dim_slice["relative_depth_percent"],
    bins=[-0.1, 25, 50, 75, 100.1],
    labels=["bottom_0_25", "lower_middle_25_50", "upper_middle_50_75", "top_75_100"]
)

# Convert categorical labels to string for better CSV / Power BI compatibility
dim_slice["slice_zone"] = dim_slice["slice_zone"].astype(str)

# Basic checks
print("dim_slice shape:", dim_slice.shape)
print("Unique slice_id:", dim_slice["slice_id"].nunique())
print("z_um range:", dim_slice["z_um"].min(), "to", dim_slice["z_um"].max())
print("relative depth range:", dim_slice["relative_depth_percent"].min(), "to", dim_slice["relative_depth_percent"].max())
print("\nSlice zone counts:")
print(dim_slice["slice_zone"].value_counts().sort_index())

display(dim_slice.head())
display(dim_slice.tail())

dim_slice shape: (435, 5)
Unique slice_id: 435
z_um range: 0.0 to 34720.0
relative depth range: 0.0 to 100.0

Slice zone counts:
slice_zone
bottom_0_25           109
lower_middle_25_50    109
top_75_100            109
upper_middle_50_75    108
Name: count, dtype: int64


,slice_id,z_um,z_mm,relative_depth_percent,slice_zone
0,0,0.0,0.00,0.000000,bottom_0_25
1,1,80.0,0.08,0.230415,bottom_0_25
2,2,160.0,0.16,0.460829,bottom_0_25
3,3,240.0,0.24,0.691244,bottom_0_25
4,4,320.0,0.32,0.921659,bottom_0_25


,slice_id,z_um,z_mm,relative_depth_percent,slice_zone
430,430,34400.0,34.40,99.078341,top_75_100
431,431,34480.0,34.48,99.308756,top_75_100
432,432,34560.0,34.56,99.539171,top_75_100
433,433,34640.0,34.64,99.769585,top_75_100
434,434,34720.0,34.72,100.000000,top_75_100


## 4. Create slice feature fact table

The `fact_slice_features` table contains quantitative measurements for each tomographic slice.

It includes:

- phase fractions,
- intensity statistics,
- pore object measurements,
- resin-like object measurements,
- heterogeneity index.

This table will be connected to `dim_slice` through `slice_id` in the Power BI data model.

In [6]:
# Create fact_slice_features table

feature_columns = [
    "slice_id",
    "pore_fraction",
    "resin_fraction",
    "matrix_fraction",
    "assigned_fraction",
    "mean_intensity",
    "std_intensity",
    "mean_intensity_pores",
    "mean_intensity_resin",
    "mean_intensity_matrix",
    "pore_object_count",
    "mean_pore_area_um2",
    "median_pore_area_um2",
    "mean_pore_equivalent_diameter_um",
    "median_pore_equivalent_diameter_um",
    "resin_object_count",
    "mean_resin_area_um2",
    "median_resin_area_um2",
    "mean_resin_equivalent_diameter_um",
    "median_resin_equivalent_diameter_um",
    "heterogeneity_index",
]

fact_slice_features = features_by_slice[feature_columns].copy()

# Basic validation
print("fact_slice_features shape:", fact_slice_features.shape)
print("Unique slice_id:", fact_slice_features["slice_id"].nunique())
print("Missing values per column:")
print(fact_slice_features.isna().sum())

display(fact_slice_features.head())

fact_slice_features shape: (435, 21)
Unique slice_id: 435
Missing values per column:
slice_id                               0
pore_fraction                          0
resin_fraction                         0
matrix_fraction                        0
assigned_fraction                      0
mean_intensity                         0
std_intensity                          0
mean_intensity_pores                   0
mean_intensity_resin                   0
mean_intensity_matrix                  0
pore_object_count                      0
mean_pore_area_um2                     0
median_pore_area_um2                   0
mean_pore_equivalent_diameter_um       0
median_pore_equivalent_diameter_um     0
resin_object_count                     0
mean_resin_area_um2                    0
median_resin_area_um2                  0
mean_resin_equivalent_diameter_um      0
median_resin_equivalent_diameter_um    0
heterogeneity_index                    0
dtype: int64


,slice_id,pore_fraction,resin_fraction,matrix_fraction,assigned_fraction,mean_intensity,std_intensity,mean_intensity_pores,mean_intensity_resin,mean_intensity_matrix,pore_object_count,mean_pore_area_um2,median_pore_area_um2,mean_pore_equivalent_diameter_um,median_pore_equivalent_diameter_um,resin_object_count,mean_resin_area_um2,median_resin_area_um2,mean_resin_equivalent_diameter_um,median_resin_equivalent_diameter_um,heterogeneity_index
0,0,0.056875,0.303065,0.622066,0.982006,162.006452,34.223109,76.737897,136.006699,182.515447,250,52352.000000,19200.0,216.142166,156.352804,592,117805.405405,12800.0,233.533527,127.66153,0.231376
1,1,0.049366,0.302342,0.638364,0.990071,163.499583,33.344875,77.197183,136.707571,182.891997,251,45258.964143,12800.0,200.100077,127.661530,769,90473.862159,12800.0,216.507405,127.66153,0.241253
2,2,0.040800,0.294249,0.657276,0.992324,165.032206,32.662106,77.162236,136.070321,183.481699,261,35972.413793,12800.0,179.854408,127.661530,755,89684.768212,12800.0,221.468784,127.66153,0.252997
3,3,0.033513,0.284570,0.673963,0.992046,166.618951,32.181359,77.149378,135.372068,184.301490,249,30971.887550,12800.0,171.086291,127.661530,704,93018.181818,12800.0,225.478737,127.66153,0.263488
4,4,0.028952,0.265491,0.697630,0.992074,168.204917,32.126347,77.017291,133.324743,185.328058,260,25624.615385,12800.0,155.146942,127.661530,672,90914.285714,12800.0,234.745590,127.66153,0.276853


In [7]:
# Quality checks for phase fractions and feature ranges

fraction_columns = [
    "pore_fraction",
    "resin_fraction",
    "matrix_fraction",
    "assigned_fraction",
]

print("Phase fraction summary:")
display(fact_slice_features[fraction_columns].describe())

print("\nObject count summary:")
display(
    fact_slice_features[
        ["pore_object_count", "resin_object_count"]
    ].describe()
)

print("\nEquivalent diameter summary:")
display(
    fact_slice_features[
        [
            "mean_pore_equivalent_diameter_um",
            "median_pore_equivalent_diameter_um",
            "mean_resin_equivalent_diameter_um",
            "median_resin_equivalent_diameter_um",
        ]
    ].describe()
)

# Check if phase fractions are within expected bounds
for col in fraction_columns:
    below_zero = (fact_slice_features[col] < 0).sum()
    above_one = (fact_slice_features[col] > 1).sum()
    print(f"{col}: below 0 = {below_zero}, above 1 = {above_one}")

# Check approximate sum of main phase fractions
fact_slice_features["phase_fraction_sum"] = (
    fact_slice_features["pore_fraction"]
    + fact_slice_features["resin_fraction"]
    + fact_slice_features["matrix_fraction"]
)

print("\nPhase fraction sum summary:")
display(fact_slice_features["phase_fraction_sum"].describe())

Phase fraction summary:


,pore_fraction,resin_fraction,matrix_fraction,assigned_fraction
count,435.000000,435.000000,435.000000,435.000000
mean,0.012330,0.204746,0.778862,0.995937
std,0.011771,0.016525,0.022066,0.001500
min,0.002225,0.172878,0.622066,0.978502
25%,0.004728,0.195336,0.769524,0.995467
50%,0.006675,0.203944,0.781761,0.996106
75%,0.017521,0.212551,0.792051,0.996718
max,0.056875,0.303065,0.819613,0.997886



Object count summary:


,pore_object_count,resin_object_count
count,435.000000,435.000000
mean,68.970115,253.862069
std,44.783880,56.395896
min,20.000000,201.000000
25%,40.000000,237.000000
50%,53.000000,246.000000
75%,86.000000,256.000000
max,296.000000,769.000000



Equivalent diameter summary:


,mean_pore_equivalent_diameter_um,median_pore_equivalent_diameter_um,mean_resin_equivalent_diameter_um,median_resin_equivalent_diameter_um
count,435.000000,435.000000,435.000000,435.000000
mean,159.359666,105.131443,418.262541,394.016616
std,33.576099,24.024819,31.500018,49.630621
min,108.929520,90.270333,216.507405,127.661530
25%,132.790319,90.270333,409.612150,380.286972
50%,155.635866,90.270333,419.431079,398.590232
75%,178.264182,127.661530,433.629186,423.405394
max,273.782623,221.116256,477.639418,477.665706


pore_fraction: below 0 = 0, above 1 = 0
resin_fraction: below 0 = 0, above 1 = 0
matrix_fraction: below 0 = 0, above 1 = 0
assigned_fraction: below 0 = 0, above 1 = 0

Phase fraction sum summary:


count    435.000000
mean       0.995937
std        0.001500
min        0.978502
25%        0.995467
50%        0.996106
75%        0.996718
max        0.997886
Name: phase_fraction_sum, dtype: float64

In [8]:
# Add segmentation coverage metrics for Power BI

fact_slice_features["phase_fraction_sum"] = (
    fact_slice_features["pore_fraction"]
    + fact_slice_features["resin_fraction"]
    + fact_slice_features["matrix_fraction"]
)

fact_slice_features["unassigned_fraction"] = 1 - fact_slice_features["phase_fraction_sum"]

# Keep unassigned_fraction within a valid range in case of tiny numerical deviations
fact_slice_features["unassigned_fraction"] = fact_slice_features["unassigned_fraction"].clip(lower=0)

print("Updated fact_slice_features shape:", fact_slice_features.shape)

print("\nUnassigned fraction summary:")
display(fact_slice_features["unassigned_fraction"].describe())

display(fact_slice_features.head())

Updated fact_slice_features shape: (435, 23)

Unassigned fraction summary:


count    435.000000
mean       0.004063
std        0.001500
min        0.002114
25%        0.003282
50%        0.003894
75%        0.004533
max        0.021498
Name: unassigned_fraction, dtype: float64

,slice_id,pore_fraction,resin_fraction,matrix_fraction,assigned_fraction,mean_intensity,std_intensity,mean_intensity_pores,mean_intensity_resin,mean_intensity_matrix,pore_object_count,mean_pore_area_um2,median_pore_area_um2,mean_pore_equivalent_diameter_um,median_pore_equivalent_diameter_um,resin_object_count,mean_resin_area_um2,median_resin_area_um2,mean_resin_equivalent_diameter_um,median_resin_equivalent_diameter_um,heterogeneity_index,phase_fraction_sum,unassigned_fraction
0,0,0.056875,0.303065,0.622066,0.982006,162.006452,34.223109,76.737897,136.006699,182.515447,250,52352.000000,19200.0,216.142166,156.352804,592,117805.405405,12800.0,233.533527,127.66153,0.231376,0.982006,0.017994
1,1,0.049366,0.302342,0.638364,0.990071,163.499583,33.344875,77.197183,136.707571,182.891997,251,45258.964143,12800.0,200.100077,127.661530,769,90473.862159,12800.0,216.507405,127.66153,0.241253,0.990071,0.009929
2,2,0.040800,0.294249,0.657276,0.992324,165.032206,32.662106,77.162236,136.070321,183.481699,261,35972.413793,12800.0,179.854408,127.661530,755,89684.768212,12800.0,221.468784,127.66153,0.252997,0.992324,0.007676
3,3,0.033513,0.284570,0.673963,0.992046,166.618951,32.181359,77.149378,135.372068,184.301490,249,30971.887550,12800.0,171.086291,127.661530,704,93018.181818,12800.0,225.478737,127.66153,0.263488,0.992046,0.007954
4,4,0.028952,0.265491,0.697630,0.992074,168.204917,32.126347,77.017291,133.324743,185.328058,260,25624.615385,12800.0,155.146942,127.661530,672,90914.285714,12800.0,234.745590,127.66153,0.276853,0.992074,0.007926


## 5. Create anomaly detection fact table

The `fact_anomaly_results` table stores the PCA coordinates and Isolation Forest outputs for each slice.

It will be used in Power BI to build:

- anomaly score profiles,
- PCA anomaly maps,
- anomaly category summaries,
- normal vs anomalous slice indicators.

In [9]:
# Create fact_anomaly_results table

anomaly_columns = [
    "slice_id",
    "PC1",
    "PC2",
    "is_anomaly",
    "anomaly_score",
    "anomaly_category",
]

fact_anomaly_results = anomaly_results[anomaly_columns].copy()

# Standardize anomaly flag for Power BI
# In the previous project, is_anomaly may be boolean or integer depending on how it was saved.
fact_anomaly_results["anomaly_flag"] = fact_anomaly_results["is_anomaly"].astype(int)

# Human-readable status for dashboard labels
fact_anomaly_results["anomaly_status"] = np.where(
    fact_anomaly_results["anomaly_flag"] == 1,
    "Anomalous",
    "Normal"
)

# Absolute anomaly score can be useful for ranking or conditional formatting
fact_anomaly_results["anomaly_score_abs"] = fact_anomaly_results["anomaly_score"].abs()

# Basic checks
print("fact_anomaly_results shape:", fact_anomaly_results.shape)
print("Unique slice_id:", fact_anomaly_results["slice_id"].nunique())

print("\nAnomaly flag counts:")
print(fact_anomaly_results["anomaly_flag"].value_counts().sort_index())

print("\nAnomaly status counts:")
print(fact_anomaly_results["anomaly_status"].value_counts())

print("\nAnomaly categories:")
print(fact_anomaly_results["anomaly_category"].value_counts())

display(fact_anomaly_results.head())

fact_anomaly_results shape: (435, 9)
Unique slice_id: 435

Anomaly flag counts:
anomaly_flag
0    413
1     22
Name: count, dtype: int64

Anomaly status counts:
anomaly_status
Normal       413
Anomalous     22
Name: count, dtype: int64

Anomaly categories:
anomaly_category
normal                                                   413
high_porosity+resin_rich+resin_fragmented                  6
high_porosity+resin_fragmented+large_pores                 5
resin_rich+resin_fragmented                                4
high_porosity+resin_rich+resin_fragmented+large_pores      3
high_porosity+large_pores                                  2
high_porosity+resin_rich+large_pores                       1
high_porosity+resin_rich                                   1
Name: count, dtype: int64


,slice_id,PC1,PC2,is_anomaly,anomaly_score,anomaly_category,anomaly_flag,anomaly_status,anomaly_score_abs
0,0,-16.003458,-2.397432,True,0.232357,high_porosity+resin_rich+resin_fragmented+larg...,1,Anomalous,0.232357
1,1,-16.159629,-4.010913,True,0.224484,high_porosity+resin_rich+resin_fragmented+larg...,1,Anomalous,0.224484
2,2,-14.789948,-4.687371,True,0.208778,high_porosity+resin_rich+resin_fragmented+larg...,1,Anomalous,0.208778
3,3,-13.171530,-4.773668,True,0.180866,high_porosity+resin_rich+resin_fragmented,1,Anomalous,0.180866
4,4,-11.460013,-4.886507,True,0.164346,high_porosity+resin_rich+resin_fragmented,1,Anomalous,0.164346


## 6. Create anomaly category summary table

This table summarizes the number and percentage of slices assigned to each anomaly category.

It is useful for:

- dashboard category cards,
- anomaly distribution plots,
- README summary tables,
- quick interpretation of the process monitoring results.

In [10]:
# Create anomaly category summary

summary_anomaly_categories = (
    fact_anomaly_results
    .groupby(["anomaly_category", "anomaly_status"], as_index=False)
    .agg(
        n_slices=("slice_id", "count"),
        mean_anomaly_score=("anomaly_score", "mean"),
        median_anomaly_score=("anomaly_score", "median"),
        mean_pc1=("PC1", "mean"),
        mean_pc2=("PC2", "mean"),
    )
)

summary_anomaly_categories["percentage_slices"] = (
    summary_anomaly_categories["n_slices"] / len(fact_anomaly_results) * 100
)

summary_anomaly_categories = summary_anomaly_categories.sort_values(
    by="n_slices",
    ascending=False
).reset_index(drop=True)

print("summary_anomaly_categories shape:", summary_anomaly_categories.shape)
display(summary_anomaly_categories)

summary_anomaly_categories shape: (8, 8)


,anomaly_category,anomaly_status,n_slices,mean_anomaly_score,median_anomaly_score,mean_pc1,mean_pc2,percentage_slices
0,normal,Normal,413,-0.132527,-0.149176,0.358672,0.025540,94.942529
1,high_porosity+resin_rich+resin_fragmented,Anomalous,6,0.121663,0.141199,-9.013104,-3.755525,1.379310
2,high_porosity+resin_fragmented+large_pores,Anomalous,5,0.007555,0.003354,-3.124369,4.318787,1.149425
3,resin_rich+resin_fragmented,Anomalous,4,0.043262,0.040408,-4.606527,-3.133913,0.919540
4,high_porosity+resin_rich+resin_fragmented+larg...,Anomalous,3,0.221873,0.224484,-15.651011,-3.698572,0.689655
5,high_porosity+large_pores,Anomalous,2,0.024762,0.024762,-3.092710,4.940128,0.459770
6,high_porosity+resin_rich+large_pores,Anomalous,1,0.002389,0.002389,-2.906999,3.804937,0.229885
7,high_porosity+resin_rich,Anomalous,1,0.004544,0.004544,-3.959415,0.337431,0.229885


## 7. Inspect Project 02 summary tables

Project 02 provides global segmentation and object-level summaries.

These tables will be used to create dashboard-ready summaries for:

- phase fraction overview,
- 2D vs 3D object count comparison,
- equivalent diameter comparison,
- 3D component summary by phase.

In [11]:
# Inspect selected Project 02 summary tables

project02_summary_names = [
    "phase_fraction_stats",
    "object_counts_2d_vs_3d",
    "diameter_2d_vs_3d",
    "components_3d_summary",
    "final_project_summary",
]

for name in project02_summary_names:
    df = tables[name].copy()
    print("=" * 100)
    print(name)
    print("Shape:", df.shape)
    print("Columns:")
    print(list(df.columns))
    display(df)

phase_fraction_stats
Shape: (3, 10)
Columns:
['Unnamed: 0', 'count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max', 'coefficient_of_variation']


,Unnamed: 0,count,mean,std,min,25%,50%,75%,max,coefficient_of_variation
0,pores_fraction,435.0,0.012330,0.011771,0.002225,0.004728,0.006675,0.017521,0.056875,0.954719
1,resin_fraction,435.0,0.208809,0.016981,0.175965,0.199675,0.207615,0.216556,0.321059,0.081323
2,material_fraction,435.0,0.778862,0.022066,0.622066,0.769524,0.781761,0.792051,0.819613,0.028331


object_counts_2d_vs_3d
Shape: (2, 4)
Columns:
['phase', 'objects_2d', 'objects_3d', 'objects_2d_per_3d_object']


,phase,objects_2d,objects_3d,objects_2d_per_3d_object
0,pores,5280,232,22.758621
1,resin,86333,1757,49.136596


diameter_2d_vs_3d
Shape: (2, 7)
Columns:
['phase', 'mean_equivalent_diameter_2d_um', 'median_equivalent_diameter_2d_um', 'std_equivalent_diameter_2d_um', 'mean_equivalent_diameter_3d_um', 'median_equivalent_diameter_3d_um', 'std_equivalent_diameter_3d_um']


,phase,mean_equivalent_diameter_2d_um,median_equivalent_diameter_2d_um,std_equivalent_diameter_2d_um,mean_equivalent_diameter_3d_um,median_equivalent_diameter_3d_um,std_equivalent_diameter_3d_um
0,pores,362.344433,270.811000,331.069805,570.020004,496.280393,305.929826
1,resin,499.255427,469.058411,225.235480,686.088833,613.377414,389.979221


components_3d_summary
Shape: (2, 16)
Columns:
['phase', 'number_of_objects', 'mean_volume_voxels', 'median_volume_voxels', 'std_volume_voxels', 'mean_volume_mm3', 'median_volume_mm3', 'mean_equivalent_diameter_um', 'median_equivalent_diameter_um', 'std_equivalent_diameter_um', 'mean_elongation_bbox', 'median_elongation_bbox', 'mean_extent_3d', 'median_extent_3d', 'mean_intensity', 'median_intensity']


,phase,number_of_objects,mean_volume_voxels,median_volume_voxels,std_volume_voxels,mean_volume_mm3,median_volume_mm3,mean_equivalent_diameter_um,median_equivalent_diameter_um,std_equivalent_diameter_um,mean_elongation_bbox,median_elongation_bbox,mean_extent_3d,median_extent_3d,mean_intensity,median_intensity
0,pores,232,536.232759,125.0,3249.312729,0.274551,0.064000,570.020004,496.280393,305.929826,1.272622,1.166667,0.435776,0.492470,61.922860,58.269509
1,resin,1757,1810.328970,236.0,54766.772829,0.926888,0.120832,686.088833,613.377414,389.979221,1.499497,1.363636,0.324337,0.322751,132.133508,130.658038


final_project_summary
Shape: (13, 4)
Columns:
['category', 'metric', 'value', 'unit']


,category,metric,value,unit
0,phase_fraction,pores_fraction_global,0.012330,fraction
1,phase_fraction,resin_fraction_global,0.208809,fraction
2,phase_fraction,material_fraction_global,0.778862,fraction
3,2d_components,pores_number_of_2d_objects,5280.000000,objects
4,2d_components,pores_median_equivalent_diameter_2d,270.811000,um
5,2d_components,resin_number_of_2d_objects,86333.000000,objects
6,2d_components,resin_median_equivalent_diameter_2d,469.058411,um
7,3d_components,pores_number_of_3d_objects,232.000000,objects
8,3d_components,pores_median_equivalent_diameter_3d,496.280393,um
9,3d_components_refined,resin_refined_number_of_3d_objects_ge50,2097.000000,objects


## 8. Create phase fraction summary table

The `summary_phase_fractions` table provides global descriptive statistics for each segmented phase.

It summarizes:

- pore fraction,
- resin-like fraction,
- matrix/material fraction.

This table will be used for global KPI cards and phase composition plots in Power BI.

In [12]:
# Create clean phase fraction summary table

phase_fraction_stats = tables["phase_fraction_stats"].copy()

# Rename first column if it came from an unnamed index-like column
if "Unnamed: 0" in phase_fraction_stats.columns:
    phase_fraction_stats = phase_fraction_stats.rename(columns={"Unnamed: 0": "phase_fraction_metric"})

summary_phase_fractions = phase_fraction_stats.copy()

# Create cleaner phase labels for dashboard use
phase_name_map = {
    "pores_fraction": "pores",
    "resin_fraction": "resin_like_regions",
    "material_fraction": "matrix",
    "matrix_fraction": "matrix",
}

summary_phase_fractions["phase"] = (
    summary_phase_fractions["phase_fraction_metric"]
    .map(phase_name_map)
    .fillna(summary_phase_fractions["phase_fraction_metric"])
)

# Reorder columns
preferred_columns = [
    "phase",
    "phase_fraction_metric",
    "count",
    "mean",
    "std",
    "min",
    "25%",
    "50%",
    "75%",
    "max",
    "coefficient_of_variation",
]

summary_phase_fractions = summary_phase_fractions[
    [col for col in preferred_columns if col in summary_phase_fractions.columns]
]

print("summary_phase_fractions shape:", summary_phase_fractions.shape)
display(summary_phase_fractions)

summary_phase_fractions shape: (3, 11)


,phase,phase_fraction_metric,count,mean,std,min,25%,50%,75%,max,coefficient_of_variation
0,pores,pores_fraction,435.0,0.012330,0.011771,0.002225,0.004728,0.006675,0.017521,0.056875,0.954719
1,resin_like_regions,resin_fraction,435.0,0.208809,0.016981,0.175965,0.199675,0.207615,0.216556,0.321059,0.081323
2,matrix,material_fraction,435.0,0.778862,0.022066,0.622066,0.769524,0.781761,0.792051,0.819613,0.028331


## 9. Create 2D vs 3D object summary table

The `summary_2d_3d_objects` table combines object counts, equivalent diameters and 3D component measurements.

It is designed to support the Power BI page focused on the difference between 2D slice-wise measurements and true 3D connected-component analysis.

In [13]:
# Load Project 02 comparison tables

object_counts = tables["object_counts_2d_vs_3d"].copy()
diameter_comparison = tables["diameter_2d_vs_3d"].copy()
components_3d_summary = tables["components_3d_summary"].copy()

# Merge object counts and diameter comparison by phase
summary_2d_3d_objects = object_counts.merge(
    diameter_comparison,
    on="phase",
    how="left"
)

# Select useful 3D metrics and merge them
components_3d_selected_columns = [
    "phase",
    "number_of_objects",
    "mean_volume_voxels",
    "median_volume_voxels",
    "mean_volume_mm3",
    "median_volume_mm3",
    "mean_equivalent_diameter_um",
    "median_equivalent_diameter_um",
    "std_equivalent_diameter_um",
    "mean_intensity",
    "median_intensity",
]

components_3d_selected_columns = [
    col for col in components_3d_selected_columns
    if col in components_3d_summary.columns
]

components_3d_selected = components_3d_summary[components_3d_selected_columns].copy()

# Rename 3D summary columns to avoid ambiguity after merge
components_3d_selected = components_3d_selected.rename(
    columns={
        "number_of_objects": "objects_3d_from_summary",
        "mean_volume_voxels": "mean_volume_3d_voxels",
        "median_volume_voxels": "median_volume_3d_voxels",
        "mean_volume_mm3": "mean_volume_3d_mm3",
        "median_volume_mm3": "median_volume_3d_mm3",
        "mean_equivalent_diameter_um": "mean_equivalent_diameter_3d_summary_um",
        "median_equivalent_diameter_um": "median_equivalent_diameter_3d_summary_um",
        "std_equivalent_diameter_um": "std_equivalent_diameter_3d_summary_um",
        "mean_intensity": "mean_intensity_3d",
        "median_intensity": "median_intensity_3d",
    }
)

summary_2d_3d_objects = summary_2d_3d_objects.merge(
    components_3d_selected,
    on="phase",
    how="left"
)

# Add interpretation-friendly ratios
summary_2d_3d_objects["object_count_reduction_factor_2d_to_3d"] = (
    summary_2d_3d_objects["objects_2d"] / summary_2d_3d_objects["objects_3d"]
)

summary_2d_3d_objects["mean_diameter_ratio_3d_to_2d"] = (
    summary_2d_3d_objects["mean_equivalent_diameter_3d_um"]
    / summary_2d_3d_objects["mean_equivalent_diameter_2d_um"]
)

summary_2d_3d_objects["median_diameter_ratio_3d_to_2d"] = (
    summary_2d_3d_objects["median_equivalent_diameter_3d_um"]
    / summary_2d_3d_objects["median_equivalent_diameter_2d_um"]
)

print("summary_2d_3d_objects shape:", summary_2d_3d_objects.shape)
print("Columns:")
print(list(summary_2d_3d_objects.columns))

display(summary_2d_3d_objects)

summary_2d_3d_objects shape: (2, 23)
Columns:
['phase', 'objects_2d', 'objects_3d', 'objects_2d_per_3d_object', 'mean_equivalent_diameter_2d_um', 'median_equivalent_diameter_2d_um', 'std_equivalent_diameter_2d_um', 'mean_equivalent_diameter_3d_um', 'median_equivalent_diameter_3d_um', 'std_equivalent_diameter_3d_um', 'objects_3d_from_summary', 'mean_volume_3d_voxels', 'median_volume_3d_voxels', 'mean_volume_3d_mm3', 'median_volume_3d_mm3', 'mean_equivalent_diameter_3d_summary_um', 'median_equivalent_diameter_3d_summary_um', 'std_equivalent_diameter_3d_summary_um', 'mean_intensity_3d', 'median_intensity_3d', 'object_count_reduction_factor_2d_to_3d', 'mean_diameter_ratio_3d_to_2d', 'median_diameter_ratio_3d_to_2d']


,phase,objects_2d,objects_3d,objects_2d_per_3d_object,mean_equivalent_diameter_2d_um,median_equivalent_diameter_2d_um,std_equivalent_diameter_2d_um,mean_equivalent_diameter_3d_um,median_equivalent_diameter_3d_um,std_equivalent_diameter_3d_um,objects_3d_from_summary,mean_volume_3d_voxels,median_volume_3d_voxels,mean_volume_3d_mm3,median_volume_3d_mm3,mean_equivalent_diameter_3d_summary_um,median_equivalent_diameter_3d_summary_um,std_equivalent_diameter_3d_summary_um,mean_intensity_3d,median_intensity_3d,object_count_reduction_factor_2d_to_3d,mean_diameter_ratio_3d_to_2d,median_diameter_ratio_3d_to_2d
0,pores,5280,232,22.758621,362.344433,270.811000,331.069805,570.020004,496.280393,305.929826,232,536.232759,125.0,0.274551,0.064000,570.020004,496.280393,305.929826,61.922860,58.269509,22.758621,1.573144,1.832571
1,resin,86333,1757,49.136596,499.255427,469.058411,225.235480,686.088833,613.377414,389.979221,1757,1810.328970,236.0,0.926888,0.120832,686.088833,613.377414,389.979221,132.133508,130.658038,49.136596,1.374224,1.307678


## 10. Export dashboard-ready tables

This section exports all cleaned and validated tables to CSV files.

These files will be used as direct inputs for:

- Power BI,
- SQLite database creation,
- README summary tables,
- portfolio documentation.

In [14]:
# Export dashboard-ready tables

export_tables = {
    "dim_slice": dim_slice,
    "fact_slice_features": fact_slice_features,
    "fact_anomaly_results": fact_anomaly_results,
    "summary_anomaly_categories": summary_anomaly_categories,
    "summary_phase_fractions": summary_phase_fractions,
    "summary_2d_3d_objects": summary_2d_3d_objects,
}

for table_name, df in export_tables.items():
    output_path = OUTPUT_DIR / f"{table_name}.csv"
    df.to_csv(output_path, index=False)
    print(f"Saved {table_name:30s} | shape: {df.shape} | {output_path}")

Saved dim_slice                      | shape: (435, 5) | C:\Users\Peter\ai-imaging-portfolio\projects\05_microct_quality_monitoring_powerbi\data\dashboard_exports\dim_slice.csv
Saved fact_slice_features            | shape: (435, 23) | C:\Users\Peter\ai-imaging-portfolio\projects\05_microct_quality_monitoring_powerbi\data\dashboard_exports\fact_slice_features.csv
Saved fact_anomaly_results           | shape: (435, 9) | C:\Users\Peter\ai-imaging-portfolio\projects\05_microct_quality_monitoring_powerbi\data\dashboard_exports\fact_anomaly_results.csv
Saved summary_anomaly_categories     | shape: (8, 8) | C:\Users\Peter\ai-imaging-portfolio\projects\05_microct_quality_monitoring_powerbi\data\dashboard_exports\summary_anomaly_categories.csv
Saved summary_phase_fractions        | shape: (3, 11) | C:\Users\Peter\ai-imaging-portfolio\projects\05_microct_quality_monitoring_powerbi\data\dashboard_exports\summary_phase_fractions.csv
Saved summary_2d_3d_objects          | shape: (2, 23) | C:\Users\

In [15]:
# Verify exported files

exported_files = sorted(OUTPUT_DIR.glob("*.csv"))

print("Exported CSV files:")
for file in exported_files:
    df_check = pd.read_csv(file)
    print(f"{file.name:35s} | shape: {df_check.shape}")

Exported CSV files:
dim_slice.csv                       | shape: (435, 5)
fact_anomaly_results.csv            | shape: (435, 9)
fact_slice_features.csv             | shape: (435, 23)
summary_2d_3d_objects.csv           | shape: (2, 23)
summary_anomaly_categories.csv      | shape: (8, 8)
summary_phase_fractions.csv         | shape: (3, 11)


## 11. Notebook summary

This notebook prepared the dashboard-ready tables for the MicroCT Quality Monitoring Power BI project.

The workflow reused outputs from previous portfolio projects:

- Project 02: 3D microCT segmentation and object-level measurements.
- Project 03: slice-wise feature extraction, PCA and Isolation Forest anomaly detection.

The exported tables are:

- `dim_slice.csv`: slice-level dimension table with depth information.
- `fact_slice_features.csv`: phase fractions, intensity statistics and object metrics by slice.
- `fact_anomaly_results.csv`: PCA coordinates, anomaly score and anomaly categories by slice.
- `summary_anomaly_categories.csv`: anomaly category distribution.
- `summary_phase_fractions.csv`: global descriptive statistics for segmented phases.
- `summary_2d_3d_objects.csv`: comparison between 2D slice-wise and 3D connected-component object measurements.

These tables provide the structured data model for the Power BI dashboard.

In [16]:
# Final notebook summary

print("Dashboard-ready tables exported successfully.\n")

for file in sorted(OUTPUT_DIR.glob("*.csv")):
    df = pd.read_csv(file)
    print(f"{file.name:35s} | rows: {df.shape[0]:4d} | columns: {df.shape[1]:2d}")

print("\nOutput folder:")
print(OUTPUT_DIR)

Dashboard-ready tables exported successfully.

dim_slice.csv                       | rows:  435 | columns:  5
fact_anomaly_results.csv            | rows:  435 | columns:  9
fact_slice_features.csv             | rows:  435 | columns: 23
summary_2d_3d_objects.csv           | rows:    2 | columns: 23
summary_anomaly_categories.csv      | rows:    8 | columns:  8
summary_phase_fractions.csv         | rows:    3 | columns: 11

Output folder:
C:\Users\Peter\ai-imaging-portfolio\projects\05_microct_quality_monitoring_powerbi\data\dashboard_exports
